In [0]:
BRONZE_ORDERS_TABLE = "swiggy.bronze.orders_raw"
BRONZE_DELIVERY_TABLE = "swiggy.bronze.delivery_raw"

SILVER_ORDERS_TABLE = "swiggy.silver.orders_clean"
SILVER_DELIVERY_TABLE = "swiggy.silver.delivery_clean"


In [0]:
from pyspark.sql.functions import col, lower, trim, upper, current_timestamp, to_timestamp, when

# Read from Bronze
bronze_orders = spark.table(BRONZE_ORDERS_TABLE)

# Apply 5 transformations using .withColumn()
silver_orders = (
    bronze_orders
    .withColumn("placed_at", to_timestamp(col("placed_at"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("payment_mode", upper(trim(col("payment_mode"))))
    .withColumn("is_valid_order", when((col("order_value") > 0) & (col("item_count") > 0), True).otherwise(False))
    .withColumn("silver_processed_at", current_timestamp())
)

silver_orders.printSchema()
print(f"Row count: {silver_orders.count()}")

bronze_delivery = spark.table(BRONZE_DELIVERY_TABLE)
silver_delivery = (
    bronze_delivery 
    .withColumn("created_at", to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("status" , lower(trim(col("status"))))
    .withColumn("is_valid_delivery" , when((col("distance_km") > 0) & (col("promised_time_min") > 0), True).otherwise(False))
    .withColumn("silver_processed_at", current_timestamp())
)

silver_delivery.printSchema()   
print(f"Row count: {silver_delivery.count()}")
   

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(SILVER_ORDERS_TABLE):
    delta_table = DeltaTable.forName(spark, SILVER_ORDERS_TABLE)

    (
        delta_table.alias("t")
        .merge(
            silver_orders.alias("s"),
            "t.order_id = s.order_id",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (silver_orders.write
                 .format("delta")
                 .saveAsTable(SILVER_ORDERS_TABLE))
    
if spark.catalog.tableExists(SILVER_DELIVERY_TABLE):
    delta_table = DeltaTable.forName(spark, SILVER_DELIVERY_TABLE)

    (
        delta_table.alias("t")
        .merge(
            silver_delivery.alias("s"),
            "t.delivery_id = s.delivery_id",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (silver_delivery.write
                 .format("delta")
                 .saveAsTable(SILVER_DELIVERY_TABLE))


In [0]:
df = spark.table(SILVER_ORDERS_TABLE)
print(f"Silver orders count: {df.count()}")
df.show(5, truncate=False)
df.printSchema()

In [0]:
df = spark.table(SILVER_DELIVERY_TABLE)
print(f"Silver delivery count: {df.count()}")
df.show(5, truncate=False)
df.printSchema()